In [ ]:
!pip install evaluate



In [ ]:
import argparse
import json
import os
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import torch
from datasets import Dataset
import evaluate
bleu_metric = evaluate.load("bleu")
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    set_seed,
)
def load_jsonl(path: str) -> Dataset:
    """
    Expect each line: {"id": "...", "input": "...", "target": "...", "label": ...}
    For generation tasks: use 'input' and 'target'
    For classification tasks: use 'text' and 'label' OR 'input' and 'label'
    """
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rows.append(json.loads(line))
    return Dataset.from_list(rows)

def preprocess_for_generation(dataset: Dataset, tokenizer, max_input_length=256, max_target_length=256, input_field="input", target_field="target"):
    def _map_fn(ex):
        inp = ex[input_field]
        tgt = ex[target_field]
        model_inputs = tokenizer(inp, truncation=True, max_length=max_input_length)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(tgt, truncation=True, max_length=max_target_length)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return dataset.map(_map_fn, batched=False, remove_columns=dataset.column_names)

def preprocess_for_classification(dataset: Dataset, tokenizer, text_field="input", label_field="label", max_length=256):
    def _map_fn(ex):
        enc = tokenizer(ex[text_field], truncation=True, max_length=max_length)
        enc["labels"] = ex[label_field]
        return enc
    return dataset.map(_map_fn, batched=False, remove_columns=dataset.column_names)
from datasets import load_metric
bleu_metric = load_metric("bleu")

def compute_generation_metrics(preds: List[str], refs: List[str]):
    tokenized_preds = [p.split() for p in preds]
    tokenized_refs = [[r.split()] for r in refs]  # bleu expects list of list of refs per example
    # compute corpus BLEU:
    bleu = bleu_metric.compute(predictions=tokenized_preds, references=tokenized_refs)
    # exact match
    exact = float(np.mean([1 if p.strip() == r.strip() else 0 for p, r in zip(preds, refs)]))
    return {"bleu": bleu["bleu"], "exact_match": exact}

def try_compute_codebleu(preds: List[str], refs: List[str], lang="python", tmp_dir="/tmp/codebleu_tmp"):
    """
    Attempts to compute CodeBLEU via `codebleu` implementation. If not present, return None.
    To install CodeBLEU: clone https://github.com/microsoft/CodeXGLUE/tree/main/Code-Code/CodeBLEU
    and pip install any dependencies. This function assumes you added codebleu to PYTHONPATH.
    """
    try:
        from codebleu import calc_code_bleu
    except Exception as e:
        print("CodeBLEU package not available:", e)
        return None
    os.makedirs(tmp_dir, exist_ok=True)
    pred_file = os.path.join(tmp_dir, "pred.txt")
    ref_file = os.path.join(tmp_dir, "ref.txt")
    with open(pred_file, "w", encoding="utf-8") as f:
        for p in preds:
            f.write(p.replace("\n", " ") + "\n")
    with open(ref_file, "w", encoding="utf-8") as f:
        for r in refs:
            f.write(r.replace("\n", " ") + "\n")
    result = calc_code_bleu(ref_file, pred_file, lang)
    return result  # dict with CodeBLEU details

def train_seq2seq(model_name: str, train_ds: Dataset, val_ds: Dataset, output_dir: str, epochs=3, batch_size=8, lr=5e-5, seed=42):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    train_tok = preprocess_for_generation(train_ds, tokenizer)
    val_tok = preprocess_for_generation(val_ds, tokenizer)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        weight_decay=0.01,
        predict_with_generate=True,
        logging_dir=os.path.join(output_dir, "logs"),
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        push_to_hub=False,
    )
    def postprocess_text(preds, labels):
        preds = [p.strip() for p in preds]
        labels = [l.strip() for l in labels]
        return preds, labels

    def compute_metrics(eval_pred):
        gen_logits, labels = eval_pred
        generated_ids = trainer.model.generate(gen_logits if isinstance(gen_logits, torch.Tensor) else gen_logits)
        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)
        metrics = compute_generation_metrics(preds, label_texts)
        return metrics

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    trainer.train()
    gen_outs = trainer.predict(val_tok, max_length=256)
    inputs = val_tok["input_ids"]
    raw_inputs = tokenizer.batch_decode(val_tok["input_ids"], skip_special_tokens=True)
    generated = []
    references = []
    for inp, ref in zip(raw_inputs, val_ds["target"]):
        input_enc = tokenizer(inp, return_tensors="pt", truncation=True).to(trainer.model.device)
        out_ids = trainer.model.generate(**input_enc, max_length=256, num_beams=4)
        pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        generated.append(pred)
        references.append(ref)
    metrics = compute_generation_metrics(generated, references)
    codebleu = try_compute_codebleu(generated, references)
    if codebleu:
        metrics["codebleu"] = codebleu
    # Save model
    trainer.save_model(output_dir)
    return {"metrics": metrics, "generated": generated, "references": references}

def train_classifier(model_name: str, train_ds: Dataset, val_ds: Dataset, label_list: List[str], output_dir: str, epochs=3, batch_size=16, lr=2e-5, seed=42):
    set_seed(seed)
    num_labels = len(label_list)
    config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    train_tok = preprocess_for_classification(train_ds, tokenizer)
    val_tok = preprocess_for_classification(val_ds, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer)
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        weight_decay=0.01,
        logging_dir=os.path.join(output_dir, "logs"),
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
    )
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        preds = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average="weighted")
        return {"accuracy": acc, "f1": f1}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    eval_res = trainer.evaluate()
    trainer.save_model(output_dir)
    return {"eval": eval_res}

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--task", choices=["generation", "classification", "both"], default="both")
    parser.add_argument("--data_train", required=True, help="Path to training JSONL")
    parser.add_argument("--data_valid", required=True, help="Path to validation JSONL")
    parser.add_argument("--output_dir", default="./outputs")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)

    train_ds = load_jsonl(args.data_train)
    valid_ds = load_jsonl(args.data_valid)

    results = {}

    if args.task in ("generation", "both"):
        print("---- Generation experiments ----")
        r1 = train_seq2seq("Salesforce/codet5-small", train_ds, valid_ds, os.path.join(args.output_dir, "codet5"), epochs=3)
        print("Codet5 results:", r1["metrics"])
        r2 = train_seq2seq("t5-small", train_ds, valid_ds, os.path.join(args.output_dir, "t5_small"), epochs=3)
        print("T5-small results:", r2["metrics"])
        results["generation"] = {"codet5": r1["metrics"], "t5_small": r2["metrics"]}

    if args.task in ("classification", "both"):
        print("---- Classification experiments ----")
        label_values = sorted(list({row["label"] for row in train_ds}))
        label_list = list(label_values)
        print("Label list:", label_list)
        r3 = train_classifier("microsoft/codebert-base", train_ds, valid_ds, label_list, os.path.join(args.output_dir, "codebert"))
        print("CodeBERT eval:", r3["eval"])
        r4 = train_classifier("bert-base-uncased", train_ds, valid_ds, label_list, os.path.join(args.output_dir, "bert_base"))
        print("BERT eval:", r4["eval"])
        results["classification"] = {"codebert": r3["eval"], "bert": r4["eval"]}


    print("==== Summary of results ====")
    print(json.dumps(results, indent=2, default=str))

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'evaluate'